# 03 - Perception review

Visually review deterministic media facts, hard-cut shot boundaries, and representative frames for one video.

Model-free: ffprobe + PySceneDetect + ffmpeg only.

In [ ]:
import sys
sys.path.insert(0, "src")

from pathlib import Path
from IPython.display import Image, display

from tiktok_analytics_factory.perception import PerceptionConfig, run_perception, evaluate_boundaries

VIDEO_PATH = Path("data/raw/<video_id>/video.mp4")  # set to the reference video from issue #2
OUTPUT_DIR = Path("data/derived/<video_id>/perception/v1")
CONFIG = PerceptionConfig()  # threshold=27.0 default

In [ ]:
manifest = run_perception(VIDEO_PATH, OUTPUT_DIR, CONFIG)

facts = manifest.media_facts
print(f"duration={facts.duration_seconds:.3f}s  {facts.width}x{facts.height} ({facts.aspect_ratio_label})")
print(f"fps={facts.fps_rational} ({facts.fps:.4f})  frames={facts.frame_count}")
print(f"vcodec={facts.video_codec}  acodec={facts.audio_codec}  sha256={facts.sha256[:16]}...")
print(f"shots detected: {len(manifest.shots.shots)}")

In [ ]:
for shot in manifest.shots.shots:
    print(f"{shot.shot_id}: {shot.start_seconds:8.3f}s -> {shot.end_seconds:8.3f}s (frames {shot.start_frame}-{shot.end_frame})")
print("\ncut boundaries:", [round(s.start_seconds, 3) for s in manifest.shots.shots if s.start_seconds > 0])

In [ ]:
# Representative frames, one per shot (midpoint timestamps)
for art in manifest.frames:
    print(f"{art.shot_id} @ {art.timestamp_seconds:.3f}s")
    display(Image(filename=art.path, width=240))

## Manual hard-cut annotation evaluation

After hand-reviewing the video, list the hard-cut times (seconds) below and evaluate the detector with tolerance ±0.30s. This measures deterministic cut detection only — not semantic scene changes.

In [ ]:
MANUAL_HARD_CUTS = [
    # e.g. 1.52,
]

detected = [s.start_seconds for s in manifest.shots.shots if s.start_seconds > 0]
if MANUAL_HARD_CUTS:
    ev = evaluate_boundaries(detected, MANUAL_HARD_CUTS, tolerance_seconds=0.30)
    print(ev.to_dict())
else:
    print("No manual annotation provided yet.")